# ETL Notebook 2: Trips Data Cleaning
Local pandas version for Windows development.
Cleans raw trips data and validates data quality.

## Imports

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
from pathlib import Path

## Configuration

In [ ]:
# Configuration
ARTIFACT_PATH = "./artifacts"

# File paths
RAW_LAYER_FILE = os.path.join(ARTIFACT_PATH, "trips_raw.parquet")
BRONZE_TABLE_FILE = os.path.join(ARTIFACT_PATH, "trips_bronze.parquet")
METRICS_FILE = os.path.join(ARTIFACT_PATH, "metrics.parquet")

# Parameters
RUN_DATE = "2025-12-15"

# Cleaning thresholds
MIN_DURATION_LIMIT = 31  # seconds
MAX_DAILY_EVENTS = 124
MAX_DURATION = 86400.0  # seconds
MAX_DISTANCE_KM = 1200.0

# Create artifact directory
os.makedirs(ARTIFACT_PATH, exist_ok=True)

print(f"Configuration:")
print(f"  Raw input: {RAW_LAYER_FILE}")
print(f"  Bronze output: {BRONZE_TABLE_FILE}")
print(f"  Run date: {RUN_DATE}")

## Step 1: Start Timer

In [ ]:
ingestion_start_ts = datetime.utcnow()
print(f"Cleaning started at: {ingestion_start_ts}")

## Step 2: Load Raw Data

In [ ]:
# Read raw trips data
raw_df = pd.read_parquet(RAW_LAYER_FILE)

# Filter by run_date if date column exists
if 'date' in raw_df.columns:
    raw_df['date'] = pd.to_datetime(raw_df['date'])
    run_date_dt = pd.to_datetime(RUN_DATE)
    raw_df = raw_df[raw_df['date'].dt.date == run_date_dt.date()]

print(f"Loaded {len(raw_df)} records from raw layer")

## Step 3: Define Cleaning Function

In [ ]:
def clean_raw_trips_with_metrics(
    df,
    run_date: str,
    min_duration_limit: int = 31,
    max_daily_events: int = 124,
    max_duration: float = 86400.0,
    max_distance_km: float = 1200.0,
):
    """
    Clean raw trips data with comprehensive metrics tracking.
    """
    metrics = {}
    df = df.copy()  # Don't modify original

    # Initial volume
    metrics["bronze_input_rows"] = len(df)

    # Drop coordinate columns
    cols_to_drop = ['start_latitude', 'start_longitude', 'end_latitude', 'end_longitude']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # Trip type filtering (remove correction trips)
    if 'trip_type' in df.columns:
        metrics["bronze_correction_trips_removed"] = (df['trip_type'] == 4).sum()
        df = df[df['trip_type'] != 4]
    else:
        metrics["bronze_correction_trips_removed"] = 0

    # Primary key null validation
    pk_null_condition = (
        df['unit_id'].isnull() |
        df['start'].isnull() |
        df['end'].isnull()
    )
    metrics["bronze_null_primary_key_rows"] = pk_null_condition.sum()
    df = df[~pk_null_condition]

    # Deduplication
    before_dedup = len(df)
    df = df.drop_duplicates(subset=['unit_id', 'start', 'end'])
    metrics["bronze_duplicate_rows_removed"] = before_dedup - len(df)

    # Start < End validation
    if 'start' in df.columns and 'end' in df.columns:
        start_after_end = (df['start'] >= df['end']).sum()
        metrics["bronze_start_after_end_rows"] = start_after_end
        df = df[df['start'] < df['end']]
    else:
        metrics["bronze_start_after_end_rows"] = 0

    # Derive duration and distance_km
    if 'start' in df.columns and 'end' in df.columns:
        df['duration'] = (df['end'] - df['start']).dt.total_seconds().astype(int)
    if 'distance' in df.columns:
        df['distance_km'] = df['distance'] / 1000.0

    # Distribution metrics
    if 'distance_km' in df.columns:
        metrics['bronze_distance_km_mean'] = df['distance_km'].mean()
        metrics['bronze_distance_km_std'] = df['distance_km'].std()
    if 'duration' in df.columns:
        metrics['bronze_duration_mean'] = df['duration'].mean()
        metrics['bronze_duration_std'] = df['duration'].std()

    # Speed validation
    if 'duration' in df.columns and 'distance_km' in df.columns and 'avg_speed' in df.columns:
        df['avg_speed_calc'] = np.where(
            df['duration'] > 0,
            df['distance_km'] / (df['duration'] / 3600.0),
            0
        )
        invalid_speed = (
            (df['avg_speed'] <= 0) |
            (df['avg_speed'] > 300) |
            (df['avg_speed'] <= df['avg_speed_calc'])
        )
        metrics["bronze_invalid_avg_speed_rows"] = invalid_speed.sum()
        df.loc[invalid_speed, 'avg_speed'] = np.nan
        df = df.drop(columns=['avg_speed_calc'])
    else:
        metrics["bronze_invalid_avg_speed_rows"] = 0

    # Duration & distance rules
    if 'duration' in df.columns and 'distance_km' in df.columns:
        rule_condition = (
            (df['duration'] > min_duration_limit) &
            (df['duration'] < max_duration) &
            (df['distance_km'] > 0) &
            (df['distance_km'] < max_distance_km)
        )
        metrics["bronze_rows_before_rule_filter"] = len(df)
        df = df[rule_condition]
        metrics["bronze_rows_after_rule_filter"] = len(df)
        metrics["bronze_rows_dropped_by_rules"] = (
            metrics["bronze_rows_before_rule_filter"] - metrics["bronze_rows_after_rule_filter"]
        )
    else:
        metrics["bronze_rows_before_rule_filter"] = len(df)
        metrics["bronze_rows_after_rule_filter"] = len(df)
        metrics["bronze_rows_dropped_by_rules"] = 0

    # Daily events per unit
    if 'unit_id' in df.columns and 'date' in df.columns:
        daily_counts = df.groupby(['unit_id', 'date']).size()
        df['daily_events'] = df.groupby(['unit_id', 'date'])['unit_id'].transform('count')
        metrics["bronze_excessive_daily_events_units"] = (df['daily_events'] >= max_daily_events).sum()
        df = df[df['daily_events'] < max_daily_events]
        df = df.drop(columns=['daily_events'])
    else:
        metrics["bronze_excessive_daily_events_units"] = 0

    # Idle time correction
    if 'idle_time' in df.columns and 'duration' in df.columns:
        idle_invalid = (
            (df['idle_time'] < 0) |
            (df['idle_time'] >= df['duration'])
        )
        metrics["bronze_idle_time_invalid_corrected"] = idle_invalid.sum()
        df.loc[idle_invalid, 'idle_time'] = np.nan
    else:
        metrics["bronze_idle_time_invalid_corrected"] = 0

    # Final volume
    metrics["bronze_output_rows"] = len(df)
    if metrics["bronze_input_rows"] > 0:
        metrics["bronze_survival_rate"] = metrics["bronze_output_rows"] / metrics["bronze_input_rows"]
    else:
        metrics["bronze_survival_rate"] = 0.0

    return df, metrics

print("Cleaning function defined")

## Step 4: Execute Cleaning

In [ ]:
# Clean the data
clean_df, metrics_dic = clean_raw_trips_with_metrics(
    raw_df,
    RUN_DATE,
    min_duration_limit=MIN_DURATION_LIMIT,
    max_daily_events=MAX_DAILY_EVENTS,
    max_duration=MAX_DURATION,
    max_distance_km=MAX_DISTANCE_KM
)

print(f"Cleaned to {len(clean_df)} records")
print(f"Survival rate: {metrics_dic['bronze_survival_rate']:.2%}")

## Step 5: Save Bronze Table

In [ ]:
clean_df.to_parquet(BRONZE_TABLE_FILE, index=False)
print(f"✓ Bronze table saved to: {BRONZE_TABLE_FILE}")
print(f"  Shape: {clean_df.shape}")

## Step 6: Record Ingestion Time & Metrics

In [ ]:
ingestion_end_ts = datetime.utcnow()
ingestion_duration_sec = (ingestion_end_ts - ingestion_start_ts).total_seconds()
metrics_dic["bronze_ingestion_duration_sec"] = ingestion_duration_sec

print(f"Cleaning duration: {ingestion_duration_sec:.2f} seconds")

## Step 7: Create Metrics Dataframe

In [ ]:
# Convert metrics dict to dataframe
metrics_rows = []
run_date_obj = pd.to_datetime(RUN_DATE).date()
created_at_ts = datetime.utcnow()

for metric_name, metric_value in metrics_dic.items():
    metrics_rows.append({
        'date': run_date_obj,
        'pipeline_stage': 'bronze',
        'metric_name': metric_name,
        'metric_value': float(metric_value) if metric_value is not None else None,
        'created_at': created_at_ts
    })

metrics_df = pd.DataFrame(metrics_rows)
print(f"Created {len(metrics_df)} metric records")

## Step 8: Save Metrics

In [ ]:
# Append to existing metrics or create new
if os.path.exists(METRICS_FILE):
    existing_metrics = pd.read_parquet(METRICS_FILE)
    metrics_df = pd.concat([existing_metrics, metrics_df], ignore_index=True)

metrics_df.to_parquet(METRICS_FILE, index=False)
print(f"✓ Metrics saved to: {METRICS_FILE}")
print(f"  Total metrics in file: {len(metrics_df)}")

## Step 9: Summary

In [ ]:
print("\n" + "="*60)
print("BRONZE CLEANING COMPLETED")
print("="*60)
print(f"Date: {RUN_DATE}")
print(f"Input records: {metrics_dic['bronze_input_rows']}")
print(f"Output records: {metrics_dic['bronze_output_rows']}")
print(f"Survival rate: {metrics_dic['bronze_survival_rate']:.2%}")
print(f"Duration: {ingestion_duration_sec:.2f} seconds")
print(f"Output file: {BRONZE_TABLE_FILE}")
print("="*60)